# Text generation example with conversational model and model quantization

This is a brief example of how to run text generation with a conversational (chat, instruct) causal language model, `pipeline`, and model quantization.

[Model quantization](https://huggingface.co/docs/accelerate/usage_guides/quantization) is a technique to reduce the computational and memory costs of running inference by representing the weights and activations with low-precision data types like 8-bit integer (int8) instead of the usual 32-bit floating point (float32).

In [1]:
!pip install --quiet --upgrade transformers accelerate bitsandbytes>0.37.0

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience. `BitsAndBytesConfig` is used for model quantization.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

Load a generative model and its tokenizer. You can substitute any other generative model name here, but note that Colab may have issues running larger models.

In [16]:
from google.colab import userdata
import torch

#MODEL_NAME = 'Qwen/Qwen3.5-0.8B'
MODEL_NAME = 'Qwen/Qwen3.5-0.8B-base'


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", quantization_config=quantization_config)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Instantiate a text generation pipeline using the tokenizer and model.

In [17]:

pipe = pipeline("text-generation",
    model=model,
    tokenizer=tokenizer
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [18]:
conversation = [{"role": "user", "content": "I want to eat more protein. Recommend me good cheap sources of it."}] # we have only one message here, but we could also have a conversation with multiple user/assistant turns as a starting point

output = pipe(conversation, max_new_tokens=150)

print("Output=",output)

for c in output[0]["generated_text"]:
  print(f"{c['role']}: {c['content']}")


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output= [{'generated_text': [{'role': 'user', 'content': 'I want to eat more protein. Recommend me good cheap sources of it.'}, {'role': 'assistant', 'content': 'Here are some good, **cheap** sources of protein that you can easily find for $1-$5 per serving:\n\n### **1. Legumes (Best Value)**\n*   **Black Beans / Lentils:** A 1 lb can of black beans or 1 can of chickpeas is a great budget option.\n*   **Chickpeas (Carried Beans):** A large container of 25-30 oz is very economical.\n*   **Pinto Beans:** A 1 lb can is a solid budget option.\n\n### **2. Nuts & Seeds (Good Value)**\n*   **Peanuts:** A 2 lb bulk box is a fantastic, cheap source.\n*'}]}]
user: I want to eat more protein. Recommend me good cheap sources of it.
assistant: Here are some good, **cheap** sources of protein that you can easily find for $1-$5 per serving:

### **1. Legumes (Best Value)**
*   **Black Beans / Lentils:** A 1 lb can of black beans or 1 can of chickpeas is a great budget option.
*   **Chickpeas (Carried

We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a high `temperature` parameter to generate more chaotic output:

In [24]:
output = pipe(conversation,
              do_sample=True,
              temperature=10.0,
              max_new_tokens=50)

for c in output[0]["generated_text"]:
  print(f"{c['role']}: {c['content']}")


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


user: I want to eat more protein. Recommend me good cheap sources of it.
assistant: Got questions in everyone needs answers with this kind that no person likes to come up and see people look good even you didn'll come out saying sorry the only issue. The most famous protein is like chicken bones just need the ground from scratch instead like I


# PART A

I want to start eating more protein in my meals and want to know what are the most cost-effective sources of protein.

prompt: "I want to eat more protein. Recommend me good cheap sources of it."

###Results with temp 0.01:
To help you find the best **cheap** protein sources, I need to clarify a few things because "cheap" can mean different things depending on your budget and location.

Here is the breakdown of the best options, categorized by how they are typically

###Results with temp 0.1:
To help you find the best **cheap** protein sources, I need to clarify a few important points about nutrition and cost:

1.  **Cost vs. Quality**: "Cheap" does not always mean "low quality." Many cheap sources are

###Results with temp 1:
To effectively increase your protein intake, it's best to focus on **whole, nutrient-dense foods** rather than just fat. High-protein foods are primarily composed of protein and healthy fats.

Here are some **cheapest and cost-effective sources
###Results with temp 10:
Improving protein maintenance involves taking care nutrition throughout **eating **a certain range. **You don’t** **you often can reduce protein through using cheap brands**!**. Because using these ways is to be in many people**
### ✅ Tips on

#Results for PART A

I selected the task of recommending the user cheap protein sources

The model performed surprisingly well with varying ranges of temperature not affecting the conclusion by that much. There was reasoning behind the statements and the recommendations were quite good.

I would say that the model seemed to perform well on this task

# PART B

Testing the base model

###Results with temp 0.01:
Great choice! Protein is essential for muscle repair, satiety, and overall health. Since you want **cheap** options, we'll focus on **affordable, high-quality sources** that are easy to find and affordable.

Here are some excellent

###Results with temp 0.1:
Great choice! Protein is essential for muscle repair, satiety, and overall health. Since you want **cheap** options, we'll focus on **affordable, high-quality sources** that are widely available in most supermarkets and local markets.

Here

###Results with temp 1:
Absolutely, I understand you want to increase protein intake for reasons like strength gains, recovery, or general health. Here's a breakdown of affordable protein-rich foods along with some tips to help you choose wisely.

## Good Cheap Protein Sources:

1

###Results with temp 10:
Got questions in everyone needs answers with this kind that no person likes to come up and see people look good even you didn'll come out saying sorry the only issue. The most famous protein is like chicken bones just need the ground from scratch instead like I

#Results for PART B
The base model seems to be almost as competent as the fine-tuned model for this task. The biggest difference is in the overall exactness of the answers and the higher temperature answers are not legible.

In [7]:
# print memory usage

import sys

def report_memory_usage(message, out=sys.stderr):
    print(f'max memory allocation {message}:', file=out)
    total = 0
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.max_memory_allocated(i)
        print(f'  cuda:{i}: {mem/2**30:.1f}G', file=out)
        total += mem
    print(f'  TOTAL: {total/2**30:.1f}G', file=out)

report_memory_usage("4bit")


max memory allocation 4bit:
  cuda:0: 0.8G
  TOTAL: 0.8G
